# Integrated Gradients for DuoT5

This notebook computes Integrated Gradients for the pairwise reranker `castorini/duot5-base-msmarco`.

Unlike the cross-encoder notebook, DuoT5 is an encoder-decoder model that directly predicts a pairwise preference. For a query `q` and two passages `(d_i, d_j)`, the model is prompted with:

`Query: q Document0: d_i Document1: d_j Relevant:`

and produces a distribution over the first decoder token. We use the pairwise decision signal

`F(q, d_i, d_j) = logit(true) - logit(false)`

as the attribution target. Positive attributions support preferring `d_i` over `d_j`; negative attributions support the competing document.


In [1]:
# -- IMPORTS --
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from captum.attr import IntegratedGradients
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

/Users/evelinalune/Documents/uni/MSc-IS/thesis/ig-thesis/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
model_name = "castorini/duot5-base-msmarco"
out_dir = Path("../outputs_t5")
pairs_file = out_dir / "pairwise_scores.parquet"

n_steps = 100
max_length = 512
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)

In [3]:
def load_pairs(path: Path):
    if path.suffix == ".parquet":
        try:
            return pd.read_parquet(path)
        except ImportError:
            fallback = path.with_suffix(".pkl")
            if fallback.exists():
                return pd.read_pickle(fallback)
            raise
    if path.suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported pair file format: {path}")

pairs_df = load_pairs(pairs_file)
print(f"Pairs loaded: {len(pairs_df)}")
print(f"Queries covered: {pairs_df['qid'].nunique()}")
pairs_df.head(3) 

Pairs loaded: 80
Queries covered: 16


,qid,query,pid_i,passage_i,score_i,pid_j,passage_j,score_j,g_score,correct_pref
0,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,4083953,Cost to Attend. The total cost to attend inclu...,0.384977,0.615020,1
1,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,2281863,"If you're paying for college, you will save li...",0.389508,0.610489,1
2,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,6262988,Undergraduate Tuition. Southern Illinois Unive...,0.232101,0.767897,1


In [4]:
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
except Exception as e:
    raise ImportError(
        "Failed to load the DuoT5 tokenizer. Install the required tokenizer dependencies, "
        "for example `pip install sentencepiece protobuf`, restart the kernel, and rerun."
    ) from e

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model = model.to(device)
model.eval()

embedding_layer = model.get_input_embeddings()
decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

true_ids = tokenizer.encode("true", add_special_tokens=False)
false_ids = tokenizer.encode("false", add_special_tokens=False)
if len(true_ids) != 1 or len(false_ids) != 1:
    raise ValueError("Expected 'true' and 'false' to map to single tokens for DuoT5 scoring.")

true_token_id = true_ids[0]
false_token_id = false_ids[0]

print(f"Loaded: {model_name}")
print(f"Device: {device}")
print(f"decoder_start_token_id: {decoder_start_token_id}")
print(f"true token id: {true_token_id}")
print(f"false token id: {false_token_id}")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loaded: castorini/duot5-base-msmarco
Device: mps
decoder_start_token_id: 0
true token id: 1176
false token id: 6136


## Tokenization Helpers

For T5 we attribute over the encoder input embeddings. Since the model input is a single prompt containing query, `Document0`, and `Document1`, we track token positions for each segment in the combined prompt.


In [5]:
def duo_input(query, doc0, doc1):
    return f"Query: {query} Document0: {doc0} Document1: {doc1} Relevant:"

def ids_to_embeds(input_ids):
    return embedding_layer(input_ids)

def _tok_len(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def tokenize_duo(query, doc0, doc1, max_length=max_length):
    text = duo_input(query, doc0, doc1)

    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        padding=False,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    query_prefix = "Query: "
    doc0_prefix = " Document0: "
    doc1_prefix = " Document1: "
    suffix = " Relevant:"

    c1 = query_prefix
    c2 = c1 + query
    c3 = c2 + doc0_prefix
    c4 = c3 + doc0
    c5 = c4 + doc1_prefix
    c6 = c5 + doc1
    c7 = c6 + suffix

    l1 = _tok_len(c1)
    l2 = _tok_len(c2)
    l3 = _tok_len(c3)
    l4 = _tok_len(c4)
    l5 = _tok_len(c5)
    l6 = _tok_len(c6)
    l7 = _tok_len(c7)

    # T5 usually appends a single </s> token.
    expected_full_len = l7 + 1
    if input_ids.shape[1] != expected_full_len:
        print(
            f"Warning: segment length mismatch for qid input. "
            f"Expected {expected_full_len}, got {input_ids.shape[1]}. "
            f"This can happen due to truncation; segment positions will be clipped."
        )

    query_positions = [pos for pos in range(l1, min(l2, input_ids.shape[1]))]
    doc0_positions = [pos for pos in range(l3, min(l4, input_ids.shape[1]))]
    doc1_positions = [pos for pos in range(l5, min(l6, input_ids.shape[1]))]

    return {
        "text": text,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "tokens": tokens,
        "query_positions": query_positions,
        "doc0_positions": doc0_positions,
        "doc1_positions": doc1_positions,
    }

## Forward and Baseline

The scalar target for IG is the first-step decoder logit margin `logit(true) - logit(false)`. This directly captures the pairwise preference strength in the DuoT5 decision.


In [6]:
def forward_duo_from_embeds(input_embeds, attention_mask):
    batch_size = input_embeds.shape[0]
    decoder_input_ids = torch.full(
        (batch_size, 1),
        decoder_start_token_id,
        dtype=torch.long,
        device=input_embeds.device,
    )

    outputs = model(
        inputs_embeds=input_embeds,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
    )

    logits = outputs.logits[:, 0, :]
    margin = logits[:, true_token_id] - logits[:, false_token_id]
    return margin

def predict_duo_pair(query, doc0, doc1):
    tok = tokenize_duo(query, doc0, doc1)
    decoder_input_ids = torch.full(
        (1, 1),
        decoder_start_token_id,
        dtype=torch.long,
        device=device,
    )

    with torch.no_grad():
        outputs = model(
            input_ids=tok["input_ids"],
            attention_mask=tok["attention_mask"],
            decoder_input_ids=decoder_input_ids,
        )

    logits = outputs.logits[:, 0, :]
    tf_logits = logits[:, [false_token_id, true_token_id]]
    true_prob = torch.softmax(tf_logits, dim=-1)[:, 1].item()
    margin = (logits[:, true_token_id] - logits[:, false_token_id]).item()
    return true_prob, margin

def make_baseline_input_ids(input_ids):
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    eos_id = tokenizer.eos_token_id
    if eos_id is not None:
        for pos, token_id in enumerate(input_ids[0].tolist()):
            if token_id == eos_id:
                baseline_ids[0, pos] = eos_id
    return baseline_ids

def make_baseline_embeds(input_ids):
    baseline_ids = make_baseline_input_ids(input_ids)
    return ids_to_embeds(baseline_ids).detach()

## Aggregate Token Attributions

T5 uses SentencePiece tokenization, so subwords are merged using the `▁` marker. Attributions are returned for the full prompt as well as the query, `Document0`, and `Document1` segments separately.


In [7]:
def merge_sentencepiece(tokens, scores):
    special_tokens = set(tokenizer.all_special_tokens)
    word_tokens, word_scores = [], []
    current_word, current_score = "", 0.0

    for token, score in zip(tokens, scores):
        if token in special_tokens:
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
                current_word, current_score = "", 0.0
            continue

        if token.startswith("▁"):
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
            current_word = token.lstrip("▁") or token
            current_score = score
        else:
            current_word += token
            current_score += score

    if current_word:
        word_tokens.append(current_word)
        word_scores.append(current_score)

    return word_tokens, np.array(word_scores)

def aggregate_span(tokens, token_scores, positions):
    span_tokens = [tokens[pos] for pos in positions if pos < len(tokens)]
    span_scores = np.array([token_scores[pos] for pos in positions if pos < len(token_scores)])
    word_tokens, word_scores = merge_sentencepiece(span_tokens, span_scores)
    return {
        "tokens": span_tokens,
        "token_scores": span_scores,
        "word_tokens": word_tokens,
        "word_scores": word_scores,
        "positions": positions,
    }

def aggregate_attributions(attributions, tok):
    token_scores = attributions[0].sum(dim=-1).detach().cpu().numpy()
    full_word_tokens, full_word_scores = merge_sentencepiece(tok["tokens"], token_scores)

    return {
        "tokens": tok["tokens"],
        "token_scores": token_scores,
        "word_tokens": full_word_tokens,
        "word_scores": full_word_scores,
        "query": aggregate_span(tok["tokens"], token_scores, tok["query_positions"]),
        "doc0": aggregate_span(tok["tokens"], token_scores, tok["doc0_positions"]),
        "doc1": aggregate_span(tok["tokens"], token_scores, tok["doc1_positions"]),
    }

## Pairwise IG

Since DuoT5 already models the pairwise decision directly, only the pairwise formulation is needed here.


In [8]:
def compute_pairwise_ig_t5(query, passage_i, passage_j):
    tok = tokenize_duo(query, passage_i, passage_j)
    input_embeds = ids_to_embeds(tok["input_ids"]).detach()
    baseline_embeds = make_baseline_embeds(tok["input_ids"])

    ig = IntegratedGradients(forward_duo_from_embeds)
    attributions, delta = ig.attribute(
        inputs=input_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(tok["attention_mask"],),
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    true_prob, margin = predict_duo_pair(query, passage_i, passage_j)

    return {
        "method": "pairwise_ig_duot5",
        "input_text": tok["text"],
        "true_prob": float(true_prob),
        "margin": float(margin),
        **aggregate_attributions(attributions, tok),
        "convergence_delta": float(delta.detach().cpu().item()) if torch.is_tensor(delta) else float(delta),
    }

## Test on One Pair

Use a pair where the aggregated DuoT5 ranking score prefers the relevant passage, then inspect the direct pairwise DuoT5 probability and the resulting attributions.


In [9]:
test_row = pairs_df[pairs_df["correct_pref"] == 1].iloc[0]

print(f"Query: {test_row['query']}")
print(f"Passage i: {test_row['passage_i'][:120]}...")
print(f"Passage j: {test_row['passage_j'][:120]}...")
print(f"Aggregated ranking g score: {test_row['g_score']:.3f}")

Query: cost of attendance eastern illinois university
Passage i: Eastern Illinois University has roughly 8,000 students. Admission is selective. Tuition is approximately $8,550 per year...
Passage j: Cost to Attend. The total cost to attend includes tuition, student fees and expenses for housing, dining, and supplies. ...
Aggregated ranking g score: 0.615


In [10]:
test_pairwise = compute_pairwise_ig_t5(
    test_row["query"],
    test_row["passage_i"],
    test_row["passage_j"],
)

print(f"Direct pairwise true probability: {test_pairwise['true_prob']:.4f}")
print(f"Direct pairwise margin: {test_pairwise['margin']:.4f}")
print(f"Convergence delta: {test_pairwise['convergence_delta']:.6f}")

for segment in ["query", "doc0", "doc1"]:
    scores = test_pairwise[segment]["word_scores"]
    words = test_pairwise[segment]["word_tokens"]
    idx = np.argsort(scores)[::-1][:10]
    print(f"\nTop-10 positive words in {segment}:")
    for i in idx:
        print(f"{words[i]:<20} {scores[i]:.4f}")

Direct pairwise true probability: 1.0000
Direct pairwise margin: 11.2487
Convergence delta: 0.268893

Top-10 positive words in query:
▁illinois            2.0096
university           0.2404
of                   -0.0037
eastern              -0.1428
attendance           -0.2274
cost                 -0.9031

Top-10 positive words in doc0:
cost-of-attendance   1.1951
$24,640              0.8634
$8,550               0.7200
$2,762.32.           0.6268
university           0.5159
$10,680              0.4155
Tuition              0.2912
Tuition              0.2888
approximately        0.2592
non-residents.       0.2295

Top-10 positive words in doc1:
Attend.              0.3452
cost                 0.2233
Attendance.          0.2102
Calculator.          0.2034
1                    0.2020
expenses.            0.1719
for                  0.1608
housing,             0.1511
about                0.1225
our                  0.1185


## Run IG on All Pairs


In [ ]:
attribution_records = []
failed_pairs = []

for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Computing DuoT5 IG"):
    try:
        pairwise_ig = compute_pairwise_ig_t5(
            row["query"],
            row["passage_i"],
            row["passage_j"],
        )

        attribution_records.append({
            "qid": row["qid"],
            "query": row["query"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "g_score": row["g_score"],
            "correct_pref": row["correct_pref"],
            "pairwise_ig": pairwise_ig,
        })
    except Exception as e:
        failed_pairs.append({
            "qid": row["qid"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "error": str(e),
        })
        print(f"Error on qid={row['qid']}, pid_i={row['pid_i']}, pid_j={row['pid_j']}: {e}")

print(f"\nSuccessfully attributed: {len(attribution_records)} pairs")
if failed_pairs:
    print(f"Failed: {len(failed_pairs)} pairs")


Computing DuoT5 IG:   0%|          | 0/80 [00:00<?, ?it/s]

In [ ]:
pw_deltas = [record["pairwise_ig"]["convergence_delta"] for record in attribution_records]
true_probs = [record["pairwise_ig"]["true_prob"] for record in attribution_records]
margins = [record["pairwise_ig"]["margin"] for record in attribution_records]

print("Convergence delta - DuoT5 pairwise IG:")
print(f" mean={np.mean(pw_deltas):.6f}, max_abs={np.max(np.abs(pw_deltas)):.6f}")

print("\nDirect pairwise true probability:")
print(f" mean={np.mean(true_probs):.4f}, min={np.min(true_probs):.4f}, max={np.max(true_probs):.4f}")

print("\nDirect pairwise margin:")
print(f" mean={np.mean(margins):.4f}, min={np.min(margins):.4f}, max={np.max(margins):.4f}")


In [ ]:
def show_top_words(record, segment="doc0", n=10):
    attr = record["pairwise_ig"][segment]
    words = attr["word_tokens"]
    scores = attr["word_scores"]

    top_pos = np.argsort(scores)[::-1][:n]
    top_neg = np.argsort(scores)[:n]

    print(f"Segment: {segment}")
    print(f"Query: {record['query']}")
    print(f"Aggregated g_score: {record['g_score']:.3f}")
    print(f"Direct DuoT5 true_prob: {record['pairwise_ig']['true_prob']:.4f}")
    print(f"Direct DuoT5 margin: {record['pairwise_ig']['margin']:.4f}")

    print(f"\nTop-{n} positive words:")
    for i in top_pos:
        print(f" {words[i]:<25} {scores[i]:.4f}")

    print(f"\nTop-{n} negative words:")
    for i in top_neg:
        print(f" {words[i]:<25} {scores[i]:.4f}")

r = attribution_records[0]
print("=" * 60)
show_top_words(r, segment="query")
print()
show_top_words(r, segment="doc0")
print()
show_top_words(r, segment="doc1")


## Save Outputs


In [ ]:
out_path = out_dir / "attributions.pkl"
with open(out_path, "wb") as f:
    pickle.dump(attribution_records, f)

print(f"Saved {len(attribution_records)} attribution records → {out_path}")

summary = pd.DataFrame([
    {
        "qid": record["qid"],
        "pid_i": record["pid_i"],
        "pid_j": record["pid_j"],
        "g_score": record["g_score"],
        "correct_pref": record["correct_pref"],
        "direct_true_prob": record["pairwise_ig"]["true_prob"],
        "direct_margin": record["pairwise_ig"]["margin"],
        "pw_ig_delta": record["pairwise_ig"]["convergence_delta"],
        "doc0_top_word": record["pairwise_ig"]["doc0"]["word_tokens"][np.argmax(record["pairwise_ig"]["doc0"]["word_scores"])] if len(record["pairwise_ig"]["doc0"]["word_tokens"]) > 0 else "",
        "doc1_top_word": record["pairwise_ig"]["doc1"]["word_tokens"][np.argmax(record["pairwise_ig"]["doc1"]["word_scores"])] if len(record["pairwise_ig"]["doc1"]["word_tokens"]) > 0 else "",
    }
    for record in attribution_records
])

summary_path = out_dir / "attributions_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary CSV → {summary_path}")
summary.head()
